In [0]:
# importação da bibliotecas

from pyspark.sql.functions import (
    col,
    to_date,
    regexp_replace,
    concat,
    lit,
    last_day,
    to_date,
    year,
    month,
    quarter,
    concat_ws,
    when,
    date_format,
    max,
    min,
    count
)

# 1. LEITURA DAS TABELAS DA CAMADA BRONZE

Nesta etapa são carregadas as tabelas persistidas na camada Bronze, que servirão como fonte para as transformações da camada Silver.

O objetivo desta camada é aplicar regras de qualidade, padronização e integração dos dados, preparando-os para consumo analítico.

In [0]:
df_cdi = spark.table("bronze_cdi")

df_selic = spark.table("bronze_selic")

df_ipca = spark.table("bronze_ipca")

In [0]:
display(df_cdi)
display(df_selic)
display(df_ipca)

data,valor,data_ingestao,arquivo_origem
01/03/2023,"13,65",2026-09-17T19:37:28.756Z,cdi_4389.csv
02/03/2023,"13,65",2026-09-17T19:37:28.756Z,cdi_4389.csv
03/03/2023,"13,65",2026-09-17T19:37:28.756Z,cdi_4389.csv
06/03/2023,"13,65",2026-09-17T19:37:28.756Z,cdi_4389.csv
07/03/2023,"13,65",2026-09-17T19:37:28.756Z,cdi_4389.csv
08/03/2023,"13,65",2026-09-17T19:37:28.756Z,cdi_4389.csv
09/03/2023,"13,65",2026-09-17T19:37:28.756Z,cdi_4389.csv
10/03/2023,"13,65",2026-09-17T19:37:28.756Z,cdi_4389.csv
13/03/2023,"13,65",2026-09-17T19:37:28.756Z,cdi_4389.csv
14/03/2023,"13,65",2026-09-17T19:37:28.756Z,cdi_4389.csv


data,valor,data_ingestao,arquivo_origem
31/12/2018,"6,50",2026-09-17T19:37:28.756Z,selic_432.csv
01/01/2019,"6,50",2026-09-17T19:37:28.756Z,selic_432.csv
02/01/2019,"6,50",2026-09-17T19:37:28.756Z,selic_432.csv
03/01/2019,"6,50",2026-09-17T19:37:28.756Z,selic_432.csv
04/01/2019,"6,50",2026-09-17T19:37:28.756Z,selic_432.csv
05/01/2019,"6,50",2026-09-17T19:37:28.756Z,selic_432.csv
06/01/2019,"6,50",2026-09-17T19:37:28.756Z,selic_432.csv
07/01/2019,"6,50",2026-09-17T19:37:28.756Z,selic_432.csv
08/01/2019,"6,50",2026-09-17T19:37:28.756Z,selic_432.csv
09/01/2019,"6,50",2026-09-17T19:37:28.756Z,selic_432.csv


data,valor,data_ingestao,arquivo_origem
07/2016,"0,52",2026-09-17T19:37:28.756Z,ipca_433.csv
08/2016,"0,44",2026-09-17T19:37:28.756Z,ipca_433.csv
09/2016,"0,08",2026-09-17T19:37:28.756Z,ipca_433.csv
10/2016,"0,26",2026-09-17T19:37:28.756Z,ipca_433.csv
11/2016,"0,18",2026-09-17T19:37:28.756Z,ipca_433.csv
12/2016,"0,30",2026-09-17T19:37:28.756Z,ipca_433.csv
01/2017,"0,38",2026-09-17T19:37:28.756Z,ipca_433.csv
02/2017,"0,33",2026-09-17T19:37:28.756Z,ipca_433.csv
03/2017,"0,25",2026-09-17T19:37:28.756Z,ipca_433.csv
04/2017,"0,14",2026-09-17T19:37:28.756Z,ipca_433.csv


# 2. ANÁLISE ESTRUTURAL DAS TABELAS BRONZE

Nesta etapa são avaliadas as estruturas das tabelas provenientes da camada Bronze.

O objetivo é identificar os tipos de dados existentes e planejar as transformações necessárias para a construção da camada Silver.

Nesta camada ocorrerão as conversões de tipos, padronização temporal e integração das séries econômicas.

In [0]:
df_cdi.printSchema()

df_selic.printSchema()

df_ipca.printSchema()


root
 |-- data: string (nullable = true)
 |-- valor: string (nullable = true)
 |-- data_ingestao: timestamp (nullable = true)
 |-- arquivo_origem: string (nullable = true)

root
 |-- data: string (nullable = true)
 |-- valor: string (nullable = true)
 |-- data_ingestao: timestamp (nullable = true)
 |-- arquivo_origem: string (nullable = true)

root
 |-- data: string (nullable = true)
 |-- valor: string (nullable = true)
 |-- data_ingestao: timestamp (nullable = true)
 |-- arquivo_origem: string (nullable = true)



In [0]:
df_cdi.show(5, truncate=False)

+----------+-----+--------------------------+--------------+
|data      |valor|data_ingestao             |arquivo_origem|
+----------+-----+--------------------------+--------------+
|01/07/2016|14,13|2026-09-17 19:37:28.756909|cdi_4389.csv  |
|04/07/2016|14,13|2026-09-17 19:37:28.756909|cdi_4389.csv  |
|05/07/2016|14,13|2026-09-17 19:37:28.756909|cdi_4389.csv  |
|06/07/2016|14,13|2026-09-17 19:37:28.756909|cdi_4389.csv  |
|07/07/2016|14,13|2026-09-17 19:37:28.756909|cdi_4389.csv  |
+----------+-----+--------------------------+--------------+
only showing top 5 rows


# 3. CONVERSÃO E PADRONIZAÇÃO DOS DADOS

A camada Silver é responsável pela padronização e preparação dos dados para consumo analítico.

Nesta etapa são realizadas transformações que convertem os dados para formatos adequados ao processamento e análise, preservando sua consistência e qualidade.

As transformações aplicadas incluem:

- Conversão de datas do formato texto para o tipo Date.
- Conversão dos valores dos indicadores para o tipo numérico.
- Padronização do formato decimal.
- Preparação das séries para integração em uma estrutura analítica única.

In [0]:
# dataframe silver do CDI
silver_cdi = (
    df_cdi
    .withColumn(
        "data_referencia",
        to_date(col("data"), "dd/MM/yyyy")
    )
    .withColumn(
        "valor_indicador",
        regexp_replace(col("valor"), ",", ".").cast("double")
    )
)

In [0]:
silver_cdi.printSchema()

root
 |-- data: string (nullable = true)
 |-- valor: string (nullable = true)
 |-- data_ingestao: timestamp (nullable = true)
 |-- arquivo_origem: string (nullable = true)
 |-- data_referencia: date (nullable = true)
 |-- valor_indicador: double (nullable = true)



In [0]:
display(
    silver_cdi.select(
        "data",
        "data_referencia",
        "valor",
        "valor_indicador"
    )
)

data,data_referencia,valor,valor_indicador
01/03/2023,2023-03-01,"13,65",13.65
02/03/2023,2023-03-02,"13,65",13.65
03/03/2023,2023-03-03,"13,65",13.65
06/03/2023,2023-03-06,"13,65",13.65
07/03/2023,2023-03-07,"13,65",13.65
08/03/2023,2023-03-08,"13,65",13.65
09/03/2023,2023-03-09,"13,65",13.65
10/03/2023,2023-03-10,"13,65",13.65
13/03/2023,2023-03-13,"13,65",13.65
14/03/2023,2023-03-14,"13,65",13.65


In [0]:
# dataframe Selic

silver_selic = (
    df_selic
    .withColumn(
        "data_referencia",
        to_date(col("data"), "dd/MM/yyyy")
    )
    .withColumn(
        "valor_indicador",
        regexp_replace(col("valor"), ",", ".").cast("double")
    )
)

In [0]:
# dataframe ipca

silver_ipca = (
    df_ipca
    .withColumn(
        "data_referencia",
        to_date(col("data"), "dd/MM/yyyy")
    )
    .withColumn(
        "valor_indicador",
        regexp_replace(col("valor"), ",", ".").cast("double")
    )
)

In [0]:
silver_selic.printSchema()

silver_ipca.printSchema()

root
 |-- data: string (nullable = true)
 |-- valor: string (nullable = true)
 |-- data_ingestao: timestamp (nullable = true)
 |-- arquivo_origem: string (nullable = true)
 |-- data_referencia: date (nullable = true)
 |-- valor_indicador: double (nullable = true)

root
 |-- data: string (nullable = true)
 |-- valor: string (nullable = true)
 |-- data_ingestao: timestamp (nullable = true)
 |-- arquivo_origem: string (nullable = true)
 |-- data_referencia: date (nullable = true)
 |-- valor_indicador: double (nullable = true)



# 4. PADRONIZAÇÃO DOS INDICADORES ECONÔMICOS

Após a validação da conversão realizada para a série CDI, o mesmo procedimento é aplicado às séries Selic e IPCA.

O objetivo desta etapa é garantir que todas as séries econômicas apresentem uma estrutura padronizada e consistente, permitindo sua integração em uma única tabela analítica na camada Silver.

As transformações realizadas incluem:

- Conversão do atributo de data para o tipo `Date`;
- Conversão dos valores dos indicadores para o tipo `Double`;
- Padronização do separador decimal;
- Criação de atributos analíticos padronizados para as três séries econômicas.

A uniformização dessas estruturas é fundamental para as etapas posteriores de consolidação, análise temporal e modelagem analítica dos dados.

In [0]:
silver_cdi.select(
    "data_referencia",
    "valor_indicador"
).show(5)

+---------------+---------------+
|data_referencia|valor_indicador|
+---------------+---------------+
|     2016-07-01|          14.13|
|     2016-07-04|          14.13|
|     2016-07-05|          14.13|
|     2016-07-06|          14.13|
|     2016-07-07|          14.13|
+---------------+---------------+
only showing top 5 rows


In [0]:
silver_selic.select(
    "data_referencia",
    "valor_indicador"
).show(5)

+---------------+---------------+
|data_referencia|valor_indicador|
+---------------+---------------+
|     2016-07-01|          14.25|
|     2016-07-02|          14.25|
|     2016-07-03|          14.25|
|     2016-07-04|          14.25|
|     2016-07-05|          14.25|
+---------------+---------------+
only showing top 5 rows


# Tratamento da Periodicidade da Série IPCA

Durante a etapa de transformação foi identificado que a série IPCA apresenta periodicidade mensal, enquanto as séries CDI e Selic são disponibilizadas em periodicidade diária.

Para permitir a integração das séries em uma estrutura analítica comum, foi necessário normalizar a informação temporal do IPCA, assumindo o último dia de cada mês como data de referência.

Exemplo:

- 07/2016 → 31/07/2016
- 08/2016 → 31/08/2016

Essa abordagem preserva a granularidade mensal da série e permite sua utilização conjunta com os demais indicadores econômicos.

In [0]:
silver_ipca.select(
    "data",
    "valor"
).show(10, truncate=False)

+-------+-----+
|data   |valor|
+-------+-----+
|07/2016|0,52 |
|08/2016|0,44 |
|09/2016|0,08 |
|10/2016|0,26 |
|11/2016|0,18 |
|12/2016|0,30 |
|01/2017|0,38 |
|02/2017|0,33 |
|03/2017|0,25 |
|04/2017|0,14 |
+-------+-----+
only showing top 10 rows


In [0]:
silver_ipca = (
    df_ipca
    .withColumn(
        "data_referencia",
        last_day(
            to_date(
                concat(
                    lit("01/"),
                    col("data")
                ),
                "dd/MM/yyyy"
            )
        )
    )
    .withColumn(
        "valor_indicador",
        regexp_replace(
            col("valor"),
            ",",
            "."
        ).cast("double")
    )
)

In [0]:
silver_ipca.select(
    "data",
    "data_referencia",
    "valor",
    "valor_indicador"
).show(10, truncate=False)

+-------+---------------+-----+---------------+
|data   |data_referencia|valor|valor_indicador|
+-------+---------------+-----+---------------+
|07/2016|2016-07-31     |0,52 |0.52           |
|08/2016|2016-08-31     |0,44 |0.44           |
|09/2016|2016-09-30     |0,08 |0.08           |
|10/2016|2016-10-31     |0,26 |0.26           |
|11/2016|2016-11-30     |0,18 |0.18           |
|12/2016|2016-12-31     |0,30 |0.3            |
|01/2017|2017-01-31     |0,38 |0.38           |
|02/2017|2017-02-28     |0,33 |0.33           |
|03/2017|2017-03-31     |0,25 |0.25           |
|04/2017|2017-04-30     |0,14 |0.14           |
+-------+---------------+-----+---------------+
only showing top 10 rows


# 5. CRIAÇÃO DA DIMENSÃO CALENDÁRIO

Com as séries devidamente padronizadas, é criada uma dimensão calendário para enriquecer as análises temporais do projeto.

Embora os indicadores já possuam um atributo de data, a utilização de uma dimensão específica permite disponibilizar informações derivadas como ano, semestre, trimestre e mês de forma padronizada.

Essa abordagem segue práticas comuns de modelagem dimensional utilizadas em ambientes analíticos e Data Warehouses.

In [0]:
dim_calendario = (
    silver_cdi
    .select("data_referencia")
    .distinct()
    .withColumn("ano", year("data_referencia"))
    .withColumn("mes", month("data_referencia"))
    .withColumn("trimestre", quarter("data_referencia"))
    .withColumn(
        "semestre",
        when(month("data_referencia") <= 6, 1).otherwise(2)
    )
    .withColumn(
        "nome_mes",
        date_format("data_referencia", "MMMM")
    )
    .withColumn(
        "ano_mes",
        concat_ws(
            "-",
            year("data_referencia"),
            month("data_referencia")
        )
    )
)

In [0]:
display(dim_calendario)

data_referencia,ano,mes,trimestre,semestre,nome_mes,ano_mes
2023-03-21,2023,3,1,1,March,2023-3
2023-05-04,2023,5,2,1,May,2023-5
2023-05-05,2023,5,2,1,May,2023-5
2023-05-29,2023,5,2,1,May,2023-5
2023-06-22,2023,6,2,1,June,2023-6
2023-08-01,2023,8,3,2,August,2023-8
2023-08-18,2023,8,3,2,August,2023-8
2023-10-04,2023,10,4,2,October,2023-10
2023-10-09,2023,10,4,2,October,2023-10
2023-10-26,2023,10,4,2,October,2023-10


# 6. CONSOLIDAÇÃO TEMPORAL DOS INDICADORES

As séries CDI e Selic são disponibilizadas em periodicidade diária, enquanto o IPCA é disponibilizado em periodicidade mensal.

Para permitir a comparação entre os indicadores em uma mesma granularidade temporal, foi adotado o último valor disponível de cada mês para as séries CDI e Selic.

Essa abordagem é amplamente utilizada no mercado financeiro e em análises econômicas, pois o fechamento do período representa o valor vigente ao final de cada mês, servindo como referência para comparações temporais, elaboração de relatórios e acompanhamento da evolução dos indicadores.

Para a série IPCA, foi utilizada a última data de cada mês como data de referência, garantindo consistência temporal entre todos os indicadores analisados.

Após essa consolidação, todas as séries passam a compartilhar a mesma granularidade mensal, possibilitando sua integração em uma única estrutura analítica para as etapas posteriores do projeto.

In [0]:
cdi_mensal_base = (
    silver_cdi
    .withColumn("ano", year("data_referencia"))
    .withColumn("mes", month("data_referencia"))
)

In [0]:
selic_mensal_base = (
    silver_selic
    .withColumn("ano", year("data_referencia"))
    .withColumn("mes", month("data_referencia"))
)

In [0]:
ultima_data_cdi = (
    cdi_mensal_base
    .groupBy("ano", "mes")
    .agg(
        max("data_referencia").alias("data_referencia")
    )
)

In [0]:
ultima_data_selic = (
    selic_mensal_base
    .groupBy("ano", "mes")
    .agg(
        max("data_referencia").alias("data_referencia")
    )
)

In [0]:
cdi_mensal = (
    ultima_data_cdi
    .join(
        cdi_mensal_base.select(
            "data_referencia",
            "valor_indicador"
        ),
        "data_referencia",
        "inner"
    )
)

In [0]:
selic_mensal = (
    ultima_data_selic
    .join(
        selic_mensal_base.select(
            "data_referencia",
            "valor_indicador"
        ),
        "data_referencia",
        "inner"
    )
)

In [0]:
display(cdi_mensal)

data_referencia,ano,mes,valor_indicador
2025-08-29,2025,8,14.9
2026-04-30,2026,4,14.4
2023-10-31,2023,10,12.65
2025-09-30,2025,9,14.9
2023-06-30,2023,6,13.65
2024-01-31,2024,1,11.65
2025-03-31,2025,3,14.15
2025-04-30,2025,4,14.15
2025-07-31,2025,7,14.9
2026-03-31,2026,3,14.65


In [0]:
display(selic_mensal)

data_referencia,ano,mes,valor_indicador
2019-10-31,2019,10,5.0
2020-06-30,2020,6,2.25
2021-01-31,2021,1,2.0
2019-04-30,2019,4,6.5
2020-01-31,2020,1,4.5
2020-04-30,2020,4,3.75
2019-01-31,2019,1,6.5
2020-09-30,2020,9,2.0
2019-02-28,2019,2,6.5
2020-02-29,2020,2,4.25


Validação dos dados:
- o primeiro mês está correto;
- o último mês está correto;
- não há meses duplicados;
- CDI e Selic realmente ficaram mensais.

In [0]:
# Primeiros Registros

cdi_mensal.orderBy("data_referencia").show(10, truncate=False)

+---------------+----+---+---------------+
|data_referencia|ano |mes|valor_indicador|
+---------------+----+---+---------------+
|2016-07-29     |2016|7  |14.13          |
|2016-08-31     |2016|8  |14.13          |
|2016-09-30     |2016|9  |14.13          |
|2016-10-31     |2016|10 |13.88          |
|2016-11-30     |2016|11 |13.88          |
|2016-12-30     |2016|12 |13.63          |
|2017-01-31     |2017|1  |12.88          |
|2017-02-24     |2017|2  |12.13          |
|2017-03-31     |2017|3  |12.13          |
|2017-04-28     |2017|4  |11.13          |
+---------------+----+---+---------------+
only showing top 10 rows


In [0]:
# últimos registros

cdi_mensal.orderBy(
    col("data_referencia").desc()
).show(10, truncate=False)

+---------------+----+---+---------------+
|data_referencia|ano |mes|valor_indicador|
+---------------+----+---+---------------+
|2026-06-30     |2026|6  |14.15          |
|2026-05-29     |2026|5  |14.4           |
|2026-04-30     |2026|4  |14.4           |
|2026-03-31     |2026|3  |14.65          |
|2026-02-27     |2026|2  |14.9           |
|2026-01-30     |2026|1  |14.9           |
|2025-12-31     |2025|12 |14.9           |
|2025-11-28     |2025|11 |14.9           |
|2025-10-31     |2025|10 |14.9           |
|2025-09-30     |2025|9  |14.9           |
+---------------+----+---+---------------+
only showing top 10 rows


In [0]:
# primeiros registros
selic_mensal.orderBy("data_referencia").show(10, truncate=False)

+---------------+----+---+---------------+
|data_referencia|ano |mes|valor_indicador|
+---------------+----+---+---------------+
|2016-07-31     |2016|7  |14.25          |
|2016-08-31     |2016|8  |14.25          |
|2016-09-30     |2016|9  |14.25          |
|2016-10-31     |2016|10 |14.0           |
|2016-11-30     |2016|11 |14.0           |
|2016-12-31     |2016|12 |13.75          |
|2017-01-31     |2017|1  |13.0           |
|2017-02-28     |2017|2  |12.25          |
|2017-03-31     |2017|3  |12.25          |
|2017-04-30     |2017|4  |11.25          |
+---------------+----+---+---------------+
only showing top 10 rows


In [0]:
# últimso registros

selic_mensal.orderBy(
    col("data_referencia").desc()
).show(10, truncate=False)

+---------------+----+---+---------------+
|data_referencia|ano |mes|valor_indicador|
+---------------+----+---+---------------+
|2026-06-30     |2026|6  |14.25          |
|2026-05-31     |2026|5  |14.5           |
|2026-04-30     |2026|4  |14.5           |
|2026-03-31     |2026|3  |14.75          |
|2026-02-28     |2026|2  |15.0           |
|2026-01-31     |2026|1  |15.0           |
|2025-12-31     |2025|12 |15.0           |
|2025-11-30     |2025|11 |15.0           |
|2025-10-31     |2025|10 |15.0           |
|2025-09-30     |2025|9  |15.0           |
+---------------+----+---+---------------+
only showing top 10 rows


In [0]:
# Primeiros registros
silver_ipca.orderBy("data_referencia").show(10, truncate=False)

+-------+-----+--------------------------+--------------+---------------+---------------+
|data   |valor|data_ingestao             |arquivo_origem|data_referencia|valor_indicador|
+-------+-----+--------------------------+--------------+---------------+---------------+
|07/2016|0,52 |2026-09-17 19:37:28.756909|ipca_433.csv  |2016-07-31     |0.52           |
|08/2016|0,44 |2026-09-17 19:37:28.756909|ipca_433.csv  |2016-08-31     |0.44           |
|09/2016|0,08 |2026-09-17 19:37:28.756909|ipca_433.csv  |2016-09-30     |0.08           |
|10/2016|0,26 |2026-09-17 19:37:28.756909|ipca_433.csv  |2016-10-31     |0.26           |
|11/2016|0,18 |2026-09-17 19:37:28.756909|ipca_433.csv  |2016-11-30     |0.18           |
|12/2016|0,30 |2026-09-17 19:37:28.756909|ipca_433.csv  |2016-12-31     |0.3            |
|01/2017|0,38 |2026-09-17 19:37:28.756909|ipca_433.csv  |2017-01-31     |0.38           |
|02/2017|0,33 |2026-09-17 19:37:28.756909|ipca_433.csv  |2017-02-28     |0.33           |
|03/2017|0

In [0]:
# últimso registros

silver_ipca.orderBy(
    col("data_referencia").desc()
).show(10, truncate=False)

+-------+-----+--------------------------+--------------+---------------+---------------+
|data   |valor|data_ingestao             |arquivo_origem|data_referencia|valor_indicador|
+-------+-----+--------------------------+--------------+---------------+---------------+
|06/2026|0,16 |2026-09-17 19:37:28.756909|ipca_433.csv  |2026-06-30     |0.16           |
|05/2026|0,58 |2026-09-17 19:37:28.756909|ipca_433.csv  |2026-05-31     |0.58           |
|04/2026|0,67 |2026-09-17 19:37:28.756909|ipca_433.csv  |2026-04-30     |0.67           |
|03/2026|0,88 |2026-09-17 19:37:28.756909|ipca_433.csv  |2026-03-31     |0.88           |
|02/2026|0,70 |2026-09-17 19:37:28.756909|ipca_433.csv  |2026-02-28     |0.7            |
|01/2026|0,33 |2026-09-17 19:37:28.756909|ipca_433.csv  |2026-01-31     |0.33           |
|12/2025|0,33 |2026-09-17 19:37:28.756909|ipca_433.csv  |2025-12-31     |0.33           |
|11/2025|0,18 |2026-09-17 19:37:28.756909|ipca_433.csv  |2025-11-30     |0.18           |
|10/2025|0

In [0]:
print("CDI:", cdi_mensal.count())
print("Selic:", selic_mensal.count())
print("IPCA:", silver_ipca.count())

CDI: 120
Selic: 120
IPCA: 120


Foi adotado o último valor disponível de cada mês para as séries CDI e Selic.

Essa abordagem é amplamente utilizada no mercado financeiro e em análises econômicas, pois representa o último valor efetivamente observado antes do encerramento de cada período de referência.

Como alguns indicadores financeiros não possuem registros em fins de semana e feriados, a data do último registro disponível pode não coincidir com o último dia do calendário. Nesses casos, considera-se a última observação efetivamente registrada.

Para viabilizar a integração dos indicadores em uma mesma granularidade temporal, será criado o atributo `ano_mes`, que representará a competência mensal de cada observação. Essa abordagem permitirá a consolidação das séries CDI, Selic e IPCA por período de referência, independentemente de pequenas diferenças nas datas de fechamento dos registros.

In [0]:
cdi_mensal = (
    cdi_mensal
    .withColumn(
        "ano_mes",
        date_format("data_referencia", "yyyy-MM")
    )
)

In [0]:
selic_mensal = (
    selic_mensal
    .withColumn(
        "ano_mes",
        date_format("data_referencia", "yyyy-MM")
    )
)

In [0]:
silver_ipca = (
    silver_ipca
    .withColumn(
        "ano_mes",
        date_format("data_referencia", "yyyy-MM")
    )
)

In [0]:
display(
    cdi_mensal.select(
        "data_referencia",
        "ano_mes",
        "valor_indicador"
    )
)

data_referencia,ano_mes,valor_indicador
2023-03-31,2023-03,13.65
2023-04-28,2023-04,13.65
2023-05-31,2023-05,13.65
2023-06-30,2023-06,13.65
2023-07-31,2023-07,13.65
2023-08-31,2023-08,13.15
2023-09-29,2023-09,12.65
2023-10-31,2023-10,12.65
2023-11-30,2023-11,12.15
2023-12-29,2023-12,11.65


In [0]:
display(
    selic_mensal.select(
        "data_referencia",
        "ano_mes",
        "valor_indicador"
    )
)

data_referencia,ano_mes,valor_indicador
2018-12-31,2018-12,6.5
2019-01-31,2019-01,6.5
2019-02-28,2019-02,6.5
2019-03-31,2019-03,6.5
2019-04-30,2019-04,6.5
2019-05-31,2019-05,6.5
2019-06-30,2019-06,6.5
2019-07-31,2019-07,6.5
2019-08-31,2019-08,6.0
2019-09-30,2019-09,5.5


In [0]:
display(
    silver_ipca.select(
        "data_referencia",
        "ano_mes",
        "valor_indicador"
    )
)

data_referencia,ano_mes,valor_indicador
2016-07-31,2016-07,0.52
2016-08-31,2016-08,0.44
2016-09-30,2016-09,0.08
2016-10-31,2016-10,0.26
2016-11-30,2016-11,0.18
2016-12-31,2016-12,0.3
2017-01-31,2017-01,0.38
2017-02-28,2017-02,0.33
2017-03-31,2017-03,0.25
2017-04-30,2017-04,0.14


# 7. INTEGRAÇÃO DAS SÉRIES ECONÔMICAS

Após a padronização e consolidação temporal dos indicadores, as séries CDI, Selic e IPCA são integradas em uma única estrutura analítica.

A integração é realizada utilizando o atributo `ano_mes`, que representa a competência temporal de cada observação.

Essa abordagem permite comparar os indicadores em uma mesma granularidade temporal, independentemente de diferenças na periodicidade de coleta ou na data exata de fechamento das séries.

A tabela resultante servirá como principal fonte de informação para as análises e para a construção da camada Gold.

In [0]:
cdi_final = (
    cdi_mensal
    .select(
        "ano_mes",
        col("valor_indicador").alias("cdi_percentual")
    )
)

In [0]:
display(cdi_final)

ano_mes,cdi_percentual
2023-03,13.65
2023-04,13.65
2023-05,13.65
2023-06,13.65
2023-07,13.65
2023-08,13.15
2023-09,12.65
2023-10,12.65
2023-11,12.15
2023-12,11.65


In [0]:
selic_final = (
    selic_mensal
    .select(
        "ano_mes",
        col("valor_indicador").alias("selic_percentual")
    )
)

In [0]:
ipca_final = (
    silver_ipca
    .select(
        "ano_mes",
        col("valor_indicador").alias("ipca_percentual")
    )
)

In [0]:
# Tabela consolidada

silver_indicadores_macro = (
    cdi_final
    .join(
        selic_final,
        "ano_mes",
        "inner"
    )
    .join(
        ipca_final,
        "ano_mes",
        "inner"
    )
)

In [0]:
display(
    silver_indicadores_macro.orderBy("ano_mes")
)

ano_mes,cdi_percentual,selic_percentual,ipca_percentual
2016-07,14.13,14.25,0.52
2016-08,14.13,14.25,0.44
2016-09,14.13,14.25,0.08
2016-10,13.88,14.0,0.26
2016-11,13.88,14.0,0.18
2016-12,13.63,13.75,0.3
2017-01,12.88,13.0,0.38
2017-02,12.13,12.25,0.33
2017-03,12.13,12.25,0.25
2017-04,11.13,11.25,0.14


In [0]:
silver_indicadores_macro.count()

120

In [0]:
silver_indicadores_macro.orderBy("ano_mes").show(5, truncate=False)

+-------+--------------+----------------+---------------+
|ano_mes|cdi_percentual|selic_percentual|ipca_percentual|
+-------+--------------+----------------+---------------+
|2016-07|14.13         |14.25           |0.52           |
|2016-08|14.13         |14.25           |0.44           |
|2016-09|14.13         |14.25           |0.08           |
|2016-10|13.88         |14.0            |0.26           |
|2016-11|13.88         |14.0            |0.18           |
+-------+--------------+----------------+---------------+
only showing top 5 rows


In [0]:
silver_indicadores_macro.orderBy(
    col("ano_mes").desc()
).show(5, truncate=False)

+-------+--------------+----------------+---------------+
|ano_mes|cdi_percentual|selic_percentual|ipca_percentual|
+-------+--------------+----------------+---------------+
|2026-06|14.15         |14.25           |0.16           |
|2026-05|14.4          |14.5            |0.58           |
|2026-04|14.4          |14.5            |0.67           |
|2026-03|14.65         |14.75           |0.88           |
|2026-02|14.9          |15.0            |0.7            |
+-------+--------------+----------------+---------------+
only showing top 5 rows


# 8. VALIDAÇÃO DA TABELA CONSOLIDADA

Após a integração dos indicadores econômicos, foram realizadas validações para verificar a consistência temporal dos dados e confirmar a manutenção do período completo de análise.

As verificações contemplaram:

- Quantidade de registros consolidados;
- Período inicial e final da série;
- Presença dos três indicadores em todos os meses;
- Consistência da chave temporal `ano_mes`.

Os resultados confirmaram a integração bem-sucedida das séries CDI, Selic e IPCA em uma única estrutura analítica mensal.


# 9. RELATÓRIO DE QUALIDADE DOS DADOS

Após a consolidação das séries econômicas, foi elaborado um relatório resumido de qualidade para validar a cobertura temporal e a quantidade de registros disponíveis para cada indicador.

O objetivo desse relatório é verificar a consistência dos dados utilizados no projeto, permitindo identificar possíveis perdas de registros durante as etapas de transformação da camada Silver.

As validações realizadas incluem:

- Quantidade total de registros;
- Data inicial da série;
- Data final da série;
- Cobertura temporal disponível para análise.

In [0]:
quality_cdi = (
    cdi_mensal
    .agg(
        count("*").alias("quantidade_registros"),
        min("data_referencia").alias("data_inicio"),
        max("data_referencia").alias("data_fim")
    )
    .withColumn("indicador", lit("CDI"))
)

In [0]:
quality_selic = (
    selic_mensal
    .agg(
        count("*").alias("quantidade_registros"),
        min("data_referencia").alias("data_inicio"),
        max("data_referencia").alias("data_fim")
    )
    .withColumn("indicador", lit("Selic"))
)

In [0]:
quality_ipca = (
    silver_ipca
    .agg(
        count("*").alias("quantidade_registros"),
        min("data_referencia").alias("data_inicio"),
        max("data_referencia").alias("data_fim")
    )
    .withColumn("indicador", lit("Ipca"))
)

In [0]:
silver_quality_report = (
    quality_cdi
    .unionByName(quality_selic)
    .unionByName(quality_ipca)
)

In [0]:
display(silver_quality_report)

quantidade_registros,data_inicio,data_fim,indicador
120,2016-07-29,2026-06-30,CDI
120,2016-07-31,2026-06-30,Selic
120,2016-07-31,2026-06-30,Ipca


# 10. PERSISTÊNCIA DA CAMADA SILVER

Após a conclusão das etapas de transformação, padronização e integração dos indicadores econômicos, as tabelas da camada Silver são persistidas no ambiente Lakehouse.

Essa etapa disponibiliza conjuntos de dados prontos para consumo analítico e para a construção da camada Gold, preservando a rastreabilidade das transformações realizadas ao longo do pipeline.

As tabelas geradas nesta camada representam a versão tratada e integrada dos dados, servindo como base para análises de negócio e simulações financeiras.

In [0]:
silver_quality_report.write \
    .mode("overwrite") \
    .saveAsTable("silver_quality_report")

In [0]:
dim_calendario.write \
    .mode("overwrite") \
    .saveAsTable("dim_calendario")

In [0]:
silver_indicadores_macro.write \
    .mode("overwrite") \
    .saveAsTable("silver_indicadores_macro")

In [0]:
spark.sql("SHOW TABLES").show(truncate=False)

+--------+------------------------+-----------+
|database|tableName               |isTemporary|
+--------+------------------------+-----------+
|default |bronze_cdi              |false      |
|default |bronze_ipca             |false      |
|default |bronze_selic            |false      |
|default |dim_calendario          |false      |
|default |silver_indicadores_macro|false      |
|default |silver_quality_report   |false      |
+--------+------------------------+-----------+



# Considerações Sobre a Consolidação Temporal dos Indicadores

Durante a construção da camada Silver, foi realizada uma análise das características temporais de cada indicador econômico utilizado no projeto.

## IPCA

A série do IPCA já é disponibilizada em periodicidade mensal pelo Banco Central. Dessa forma, não foi necessária nenhuma agregação adicional dos valores. Apenas foi realizado o tratamento da data de referência para padronização da estrutura temporal da série.

## Selic

A série da Selic é disponibilizada em periodicidade diária. Para permitir sua integração com os demais indicadores em uma mesma granularidade temporal, foi adotado o último valor disponível de cada mês.

Essa abordagem é amplamente utilizada no mercado financeiro e em análises econômicas, pois representa o valor vigente ao encerramento de cada período de referência.

## CDI

A série do CDI também é disponibilizada em periodicidade diária. Assim como realizado para a Selic, foi utilizado o último valor disponível de cada mês para compor a estrutura analítica mensal da camada Silver.

Entretanto, é importante destacar que o CDI possui aplicação direta em operações financeiras indexadas e pode demandar cálculos de acumulação para representar corretamente seu impacto ao longo do tempo.

Por esse motivo, eventuais cálculos de taxa mensal equivalente e acumulações financeiras serão realizados na camada Gold, etapa responsável pela aplicação das regras de negócio e simulações de financiamento.

## Integração dos Indicadores

Como indicadores financeiros podem não possuir registros em fins de semana e feriados, as datas de fechamento dos registros podem não coincidir exatamente com o último dia do calendário.

Para viabilizar a consolidação dos dados, foi criado o atributo `ano_mes`, representando a competência mensal de cada observação. A integração entre CDI, Selic e IPCA foi realizada utilizando essa chave temporal, garantindo comparabilidade entre os indicadores independentemente de pequenas diferenças nas datas de referência.